In [ ]:
from bs4 import BeautifulSoup
import re

def extract_article_text(file_path):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        html = f.read()

    soup = BeautifulSoup(html, "html.parser")

    # Remove obvious noise
    for tag in soup(["script", "style", "noscript", "header", "footer", "nav", "aside", "form"]):
        tag.decompose()

    # First try common article containers
    candidates = [
        soup.find("article"),
        soup.find("main"),
        soup.find(attrs={"role": "main"}),
        soup.find("div", class_=re.compile(r"article|content|post|story", re.I)),
    ]

    main_content = next((c for c in candidates if c is not None), soup)

    text = main_content.get_text(separator=" ", strip=True)
    text = re.sub(r"\s+", " ", text)

    return text

# Example
text = extract_article_text("/content/24_Source_1.html")
print(text[:3000])

In [ ]:
import os
import re
from bs4 import BeautifulSoup

def extract_article_text(file_path):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        html = f.read()

    soup = BeautifulSoup(html, "html.parser")

    # ❌ Remove junk elements
    for tag in soup(["script", "style", "noscript", "header", "footer", "nav", "aside", "form"]):
        tag.decompose()

    # 🎯 Try to find main article content
    candidates = [
        soup.find("article"),
        soup.find("main"),
        soup.find(attrs={"role": "main"}),
        soup.find("div", class_=re.compile(r"article|content|post|story|body", re.I)),
    ]

    main_content = next((c for c in candidates if c is not None), soup)

    # 🧹 Extract and clean text
    text = main_content.get_text(separator=" ", strip=True)
    text = re.sub(r"\s+", " ", text)

    return text


# =========================
# 📂 Process entire folder
# =========================
input_folder = "articles_html"
output_folder = "clean_text"

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if filename.endswith(".html"):
        file_path = os.path.join(input_folder, filename)

        try:
            clean_text = extract_article_text(file_path)

            # Save cleaned text
            output_file = os.path.join(output_folder, filename.replace(".html", ".txt"))
            with open(output_file, "w", encoding="utf-8") as f:
                f.write(clean_text)

            print(f"✅ Processed: {filename}")

        except Exception as e:
            print(f"❌ Error with {filename}: {e}")

print("\n🎉 Done! Clean text saved in 'clean_text/' folder.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
print(os.listdir("/content/drive/MyDrive"))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = "/content/drive/MyDrive/NLP/roberta_model"

model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

model.eval()

In [ ]:
import torch

text = "This campaign used misinformation to influence voters"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=256
)

with torch.no_grad():
    outputs = model(**inputs)

pred_id = torch.argmax(outputs.logits, dim=1).item()
pred_label = model.config.id2label[pred_id]

print("Prediction:", pred_label)